In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsmartdata202611";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

True

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
--DROP TABLE IF EXISTS catalog_au.bronze.movie;

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.movie (
movieId integer,
title string,
budget DOUBLE,
homePage string,
popularity double,
yearReleaseDate INTEGER,
releaseDate DATE,
revenue DOUBLE,
durationTime INTEGER,
movieStatus STRING,
voteAverage DECIMAL,
voteCount INTEGER
)
USING DELTA
PARTITIONED BY (releaseDate)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/movie"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.movie_cast (
movieId integer,
personId integer,
characterName STRING,
genderId integer,
castOrder INTEGER
)
USING DELTA
--PARTITIONED BY (race_year)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/movie_cast"

In [0]:
--CREATE TABLE IF NOT EXISTS catalog_au.bronze.movie_gender (
  --movieId integer,
  --genreId integer
--)
--USING DELTA
--LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/movie_gender"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.actors (
  personId integer,
  personname string,
  lastname STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/actors"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.movie_transformed (
  movieId integer,
  title STRING,
  budget DOUBLE,
  yearReleaseDate INTEGER,
  releaseDate DATE,
  durationTime string,
  characterName STRING,
  personId integer,
  genderId integer,
  personname string,
  lastname STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/movie_transformed"

###Tablas Golden

In [0]:
drop table if exists catalog_au.golden.movie_transformed

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.movie_transformed (
  movieId integer,
  title long,
  budget DOUBLE,
  yearReleaseDate integer,
  genderId string,
  durationTime INTEGER,
  characterName STRING
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/movie_transformed"